# Weekly Store Sales Forecasting

**Project 2 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **weekly department-level sales** for 45 Walmart stores across the United States. The dataset comes from the Kaggle **Walmart Recruiting – Store Sales Forecasting** competition and covers February 2010 through October 2012.

Walmart releases supplementary data on store types, sizes, and economic indicators (temperature, fuel price, CPI, unemployment) per store per week. The 45 stores span multiple regions and formats, creating a rich multi-series forecasting challenge.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting |
| **Target variable** | `Weekly_Sales` (department-level weekly revenue) |
| **Date column** | `Date` |
| **Frequency** | Weekly (`W`) |
| **Primary TS library** | MLForecast |
| **Kaggle competition** | `walmart-recruiting-store-sales-forecasting` |

## Learning Objectives

By completing this notebook you will learn how to:

1. Download and merge a **multi-file retail dataset** (sales + stores + features)
2. Handle **holiday markdown effects** unique to Walmart (Super Bowl, Labor Day, Thanksgiving, Christmas)
3. Aggregate department-level data to the store level for clearer signal
4. Detect and handle **negative department sales** (markdown adjustments / returns)
5. Engineer features appropriate for weekly frequency: lags at 1, 4, 13, 52 weeks
6. Build naive and seasonal-naive (52-week) baselines
7. Benchmark regressors via LazyPredict on lag-feature tabular view
8. Run FLAML AutoML with a time budget
9. Train MLForecast models (LightGBM, XGBoost) with built-in lag/rolling features
10. Select top 3 models from holdout performance
11. Analyze **weighted MAE** (the competition's actual metric, which up-weights holiday weeks 5×)

## Problem Statement

Given ~420K weekly department-sales records across 45 Walmart stores and 81 departments, **forecast weekly sales for the next 8 weeks**. The competition uses **Weighted Mean Absolute Error (WMAE)** where holiday weeks receive 5× weight.

For clarity, this notebook primarily works at the **total-store aggregate** level (sum across all departments and stores per week). Section 27 discusses scaling to the full panel.

## Why This Project Matters

- **Retail demand planning**: Walmart's supply chain depends on accurate weekly forecasts to position inventory across distribution centers.
- **Markdown optimization**: Temporary price reductions (markdowns) around holidays can dramatically shift demand. Understanding their effect is essential for pricing strategy.
- **Labor scheduling**: Weekly sales drive staffing models for 2.3M+ Walmart associates.
- **Economic sensitivity**: Store-level CPI, unemployment, and fuel prices capture macro-economic impacts on consumer spending — a textbook application of external regressors.

## Dataset Overview

The competition provides three CSV files:

| File | Rows | Description |
|------|------|-------------|
| `train.csv` | ~421 K | Weekly department sales: `Store`, `Dept`, `Date`, `Weekly_Sales`, `IsHoliday` |
| `stores.csv` | 45 | Store metadata: `Store`, `Type` (A/B/C), `Size` (sq ft) |
| `features.csv` | ~8,190 | Weekly economic features per store: `Temperature`, `Fuel_Price`, `MarkDown1-5`, `CPI`, `Unemployment`, `IsHoliday` |

### Key columns
- **`Weekly_Sales`** — target; can be negative due to markdown/return adjustments
- **`IsHoliday`** — binary; 1 for Super Bowl, Labor Day, Thanksgiving, Christmas weeks
- **`MarkDown1-5`** — promotional markdown dollar amounts (many NaN before Nov 2011)
- **`Type`** — store type: A (large), B (medium), C (small)
- **`Size`** — store floor area in sq ft; strong predictor

### Known quality issues
- MarkDown columns are NaN before November 2011 (they didn't exist yet)
- Some departments have negative `Weekly_Sales` (return/markdown adjustments)
- CPI and Unemployment data have gaps in some stores
- Holiday calendar is US-specific (Super Bowl Sunday, Labor Day, Thanksgiving, Christmas)

## Dataset Source & License Notes

- **Kaggle competition**: [Walmart Recruiting - Store Sales Forecasting](https://www.kaggle.com/competitions/walmart-recruiting-store-sales-forecasting)
- **License**: Competition-specific (Kaggle Competition Rules)
- **Provider**: Walmart Inc. (via Kaggle)
- **Usage**: Educational purposes only

## Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "mlforecast", "lightgbm", "xgboost",
    "statsmodels", "scipy", "window-ops",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
import lightgbm as lgb
import xgboost as xgb

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from scipy import stats

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Weekly Store Sales Forecasting"
KAGGLE_SLUG  = "walmart-recruiting-store-sales-forecasting"

TARGET  = "Weekly_Sales"
DATE    = "Date"
FREQ    = "W"

SEASON_LENGTH    = 52     # annual cycle for weekly data
FORECAST_HORIZON = 8      # 8-week forecast
TEST_WEEKS       = FORECAST_HORIZON
VAL_WEEKS        = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 120    # seconds

# Holiday weight for WMAE (competition metric)
HOLIDAY_WEIGHT = 5

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} weeks  |  Holiday weight: {HOLIDAY_WEIGHT}x")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False

if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    print("WARNING: No Kaggle credentials found!")
    print("See: https://www.kaggle.com/settings -> Create New Token")
else:
    print("Ready to download.")

## Dataset Download & Loading

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    print("Falling back to kaggle CLI...")
    os.makedirs("data", exist_ok=True)
    os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = sorted(data_path.rglob("*.csv"))
for f in csv_files:
    print(f"  {f.name:30s}  {f.stat().st_size / 1e6:7.2f} MB")

### Load each CSV

In [ ]:
def _find(name):
    matches = [f for f in csv_files if name in f.name.lower()]
    return matches[0] if matches else None

train_raw = pd.read_csv(_find("train"), parse_dates=["Date"])
stores    = pd.read_csv(_find("stores"))
features  = pd.read_csv(_find("features"), parse_dates=["Date"])

print(f"train_raw : {train_raw.shape}  -- {train_raw['Date'].min().date()} to {train_raw['Date'].max().date()}")
print(f"stores    : {stores.shape}")
print(f"features  : {features.shape}")

train_raw.head()

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)

assert "Weekly_Sales" in train_raw.columns, "Missing 'Weekly_Sales' column!"
assert "Date" in train_raw.columns, "Missing 'Date' column!"

print(f"\n[train.csv]")
print(f"  Shape          : {train_raw.shape[0]:,} rows x {train_raw.shape[1]} cols")
print(f"  Date range     : {train_raw['Date'].min().date()} to {train_raw['Date'].max().date()}")
print(f"  Unique stores  : {train_raw['Store'].nunique()}")
print(f"  Unique depts   : {train_raw['Dept'].nunique()}")
print(f"  Missing sales  : {train_raw['Weekly_Sales'].isna().sum()}")
print(f"  Negative sales : {(train_raw['Weekly_Sales'] < 0).sum()} ({(train_raw['Weekly_Sales']<0).mean()*100:.2f}%)")
print(f"  Holiday weeks  : {train_raw['IsHoliday'].sum()} ({train_raw['IsHoliday'].mean()*100:.1f}%)")

print(f"\n[stores.csv]")
print(f"  Type distribution: {stores['Type'].value_counts().to_dict()}")
print(f"  Size range: {stores['Size'].min():,} to {stores['Size'].max():,} sq ft")

print(f"\n[features.csv]")
md_cols = [c for c in features.columns if "MarkDown" in c]
for c in md_cols:
    pct_na = features[c].isna().mean() * 100
    print(f"  {c} NaN: {pct_na:.1f}%")
print(f"  CPI NaN: {features['CPI'].isna().mean()*100:.1f}%")
print(f"  Unemployment NaN: {features['Unemployment'].isna().mean()*100:.1f}%")

print("\nValidation complete.")

## Exploratory Data Analysis

We aggregate to **total weekly sales** across all stores and departments for a clear top-level view.

In [ ]:
# Merge train with stores and features
train_merged = train_raw.merge(stores, on="Store", how="left")
train_merged = train_merged.merge(features, on=["Store", "Date", "IsHoliday"], how="left")

# Aggregate to total weekly sales
weekly = (
    train_merged
    .groupby("Date")
    .agg(
        Weekly_Sales=("Weekly_Sales", "sum"),
        IsHoliday=("IsHoliday", "max"),
        Temperature=("Temperature", "mean"),
        Fuel_Price=("Fuel_Price", "mean"),
        CPI=("CPI", "mean"),
        Unemployment=("Unemployment", "mean"),
    )
    .reset_index()
    .sort_values("Date")
    .reset_index(drop=True)
)

print(f"Aggregated weekly series: {len(weekly)} weeks")
print(f"Date range: {weekly['Date'].min().date()} to {weekly['Date'].max().date()}")
weekly.head()

In [ ]:
# Full time-series plot with holiday markers
fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Weekly_Sales"], name="Total Weekly Sales",
                         line=dict(color="blue")))

holidays = weekly[weekly["IsHoliday"] == 1]
fig.add_trace(go.Scatter(x=holidays["Date"], y=holidays["Weekly_Sales"], mode="markers",
                         name="Holiday Week", marker=dict(color="red", size=10, symbol="star")))

fig.update_layout(title="Walmart -- Total Weekly Sales (45 Stores)", template="plotly_white",
                  xaxis_title="Date", yaxis_title="Total Weekly Sales ($)")
fig.show()

### Store type comparison

Walmart stores come in three types: A (supercenters), B (discount stores), C (neighborhood markets). Type A stores are much larger and generate more revenue.

In [ ]:
store_weekly = (
    train_merged
    .groupby(["Date", "Type"])
    .agg(Weekly_Sales=("Weekly_Sales", "sum"))
    .reset_index()
)

fig = px.line(store_weekly, x="Date", y="Weekly_Sales", color="Type",
              title="Weekly Sales by Store Type (A/B/C)")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Sales distribution by store size
store_totals = train_merged.groupby("Store").agg(
    Total_Sales=("Weekly_Sales", "sum"),
    Size=("Size", "first"),
    Type=("Type", "first"),
).reset_index()

fig = px.scatter(store_totals, x="Size", y="Total_Sales", color="Type",
                 hover_data=["Store"], title="Total Sales vs Store Size (sq ft)",
                 trendline="ols")
fig.update_layout(template="plotly_white")
fig.show()

corr = store_totals[["Size", "Total_Sales"]].corr().iloc[0, 1]
print(f"Correlation (Size vs Total_Sales): {corr:.3f}")

### Holiday effects — Super Bowl, Thanksgiving, Christmas

The competition specifically flags four US holidays. Let's see their impact on aggregate sales.

In [ ]:
# Identify the specific holidays
holiday_dates = features[features["IsHoliday"] == True]["Date"].unique()
weekly["holiday_name"] = "Non-Holiday"

# Approximate holiday identification from dates
for d in weekly["Date"]:
    m, day = d.month, d.day
    if m == 2 and 2 <= day <= 12:  # Super Bowl (early Feb)
        weekly.loc[weekly["Date"] == d, "holiday_name"] = "Super Bowl"
    elif m == 9 and 1 <= day <= 10:  # Labor Day (early Sep)
        weekly.loc[weekly["Date"] == d, "holiday_name"] = "Labor Day"
    elif m == 11 and 20 <= day <= 30:  # Thanksgiving (late Nov)
        weekly.loc[weekly["Date"] == d, "holiday_name"] = "Thanksgiving"
    elif m == 12 and 22 <= day <= 31:  # Christmas
        weekly.loc[weekly["Date"] == d, "holiday_name"] = "Christmas"

fig = px.box(weekly, x="holiday_name", y="Weekly_Sales",
             category_orders={"holiday_name": ["Non-Holiday", "Super Bowl", "Labor Day", "Thanksgiving", "Christmas"]},
             title="Sales Distribution by Holiday")
fig.update_layout(template="plotly_white")
fig.show()

print("Mean weekly sales by holiday:")
print(weekly.groupby("holiday_name")["Weekly_Sales"].mean().to_string())

In [ ]:
# Seasonal decomposition
ts = weekly.set_index("Date")["Weekly_Sales"].asfreq("W").interpolate()

decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title="Observed")
decomp.trend.plot(ax=axes[1], title="Trend")
decomp.seasonal.plot(ax=axes[2], title="Annual Seasonal (52-week)")
decomp.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

In [ ]:
# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts.dropna(), lags=60, ax=axes[0], title="ACF -- Weekly Sales")
plot_pacf(ts.dropna(), lags=60, ax=axes[1], title="PACF -- Weekly Sales")
plt.tight_layout()
plt.show()

In [ ]:
# Stationarity test
adf = adfuller(ts.dropna())
print("Augmented Dickey-Fuller Test:")
print(f"  ADF Statistic : {adf[0]:.4f}")
print(f"  p-value       : {adf[1]:.6f}")
for k, v in adf[4].items():
    print(f"  Critical ({k}): {v:.4f}")
print(f"\nResult: {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'} at 5% significance")

## Target Analysis

In [ ]:
print("Target Statistics (total weekly sales):")
desc = weekly["Weekly_Sales"].describe()
print(desc.to_string())
print(f"\nSkewness : {weekly['Weekly_Sales'].skew():.3f}")
print(f"Kurtosis : {weekly['Weekly_Sales'].kurtosis():.3f}")

Q1, Q3 = weekly["Weekly_Sales"].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = weekly[(weekly["Weekly_Sales"] < Q1 - 1.5*IQR) | (weekly["Weekly_Sales"] > Q3 + 1.5*IQR)]
print(f"\nOutliers (IQR): {len(outliers)} weeks ({len(outliers)/len(weekly)*100:.1f}%)")
if len(outliers):
    print(f"  Dates: {outliers['Date'].dt.date.tolist()}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(weekly["Weekly_Sales"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Weekly Sales")
axes[1].boxplot(weekly["Weekly_Sales"].dropna())
axes[1].set_title("Box Plot")
pd.plotting.lag_plot(weekly["Weekly_Sales"], lag=1, ax=axes[2])
axes[2].set_title("Lag-1 Plot (1 week)")
plt.tight_layout()
plt.show()

## Train / Validation / Test Split Strategy

Temporal split — no shuffling. We hold out the last 8 weeks for test and the prior 8 weeks for validation.

In [ ]:
ts_df = weekly[["Date", "Weekly_Sales", "IsHoliday"]].rename(
    columns={"Date": "ds", "Weekly_Sales": "y"}
).copy()

n = len(ts_df)
ts_train = ts_df.iloc[: n - TEST_WEEKS - VAL_WEEKS].copy()
ts_val   = ts_df.iloc[n - TEST_WEEKS - VAL_WEEKS : n - TEST_WEEKS].copy()
ts_test  = ts_df.iloc[n - TEST_WEEKS :].copy()

print(f"Train : {len(ts_train)} weeks  ({ts_train['ds'].min().date()} to {ts_train['ds'].max().date()})")
print(f"Val   : {len(ts_val)} weeks  ({ts_val['ds'].min().date()} to {ts_val['ds'].max().date()})")
print(f"Test  : {len(ts_test)} weeks  ({ts_test['ds'].min().date()} to {ts_test['ds'].max().date()})")

assert ts_train["ds"].max() < ts_val["ds"].min(), "Train/val overlap!"
assert ts_val["ds"].max()   < ts_test["ds"].min(), "Val/test overlap!"
print("\nNo temporal overlap -- split is clean.")

ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"],   y=ts_val["y"],   name="Validation", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"],  y=ts_test["y"],  name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Train / Val / Test Split", template="plotly_white")
fig.show()

## Preprocessing Strategy

For weekly aggregate data:
1. **Missing weeks** — interpolate calendar gaps (Walmart rarely misses a week)
2. **Negative aggregated sales** — extremely unlikely at aggregate; clip if present
3. **Feature imputation** — MarkDown columns are NaN before Nov 2011; fill with 0
4. **No scaling** — tree-based models handle raw values

In [ ]:
def preprocess_ts(df):
    out = df.copy()
    out = out.drop_duplicates(subset=["ds"], keep="last").sort_values("ds").reset_index(drop=True)
    if out["y"].isna().any():
        n_miss = out["y"].isna().sum()
        out["y"] = out["y"].interpolate("linear")
        print(f"  Interpolated {n_miss} missing values")
    if (out["y"] < 0).any():
        n_neg = (out["y"] < 0).sum()
        out.loc[out["y"] < 0, "y"] = 0
        print(f"  Clipped {n_neg} negatives to 0")
    return out

ts_train    = preprocess_ts(ts_train)
ts_val      = preprocess_ts(ts_val)
ts_test     = preprocess_ts(ts_test)
ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

print(f"\nClean sizes -- train: {len(ts_train)}, val: {len(ts_val)}, test: {len(ts_test)}")

## Feature Engineering

Features for the **tabular modeling** path. Weekly frequency means lags at 1, 4, 13, 52 weeks. Rolling stats use shifted values to avoid leakage.

Feature groups:
- **Lags**: 1, 2, 4, 8, 13, 52 weeks
- **Rolling stats**: 4-week and 13-week mean/std
- **Calendar**: week of year, month, quarter, is_year_end
- **External**: temperature, fuel price, CPI, unemployment (lagged), holiday flag

In [ ]:
def create_features(df, weekly_ext):
    out = df.copy()

    # Lag features  
    for lag in [1, 2, 4, 8, 13, 52]:
        out[f"lag_{lag}"] = out["y"].shift(lag)

    # Rolling statistics (shifted to avoid leakage)
    shifted = out["y"].shift(1)
    for w in [4, 13]:
        out[f"rmean_{w}"] = shifted.rolling(w).mean()
        out[f"rstd_{w}"]  = shifted.rolling(w).std()
    out["rmin_4"]  = shifted.rolling(4).min()
    out["rmax_4"]  = shifted.rolling(4).max()

    # Calendar features
    out["week"]       = out["ds"].dt.isocalendar().week.astype(int)
    out["month"]      = out["ds"].dt.month
    out["quarter"]    = out["ds"].dt.quarter
    out["is_year_end"] = ((out["month"] == 12) & (out["week"] >= 50)).astype(int)

    # External features from weekly aggregate
    ext = weekly_ext[["Date", "Temperature", "Fuel_Price", "CPI", "Unemployment", "IsHoliday"]].rename(
        columns={"Date": "ds"}
    )
    out = out.merge(ext, on="ds", how="left")
    
    # Lag external features to avoid look-ahead
    for col in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
        out[f"{col}_lag1"] = out[col].shift(1)
        out.drop(columns=[col], inplace=True)
    
    # Fill NaN in external columns
    for col in out.columns:
        if out[col].isna().any() and col not in ["ds", "y"]:
            out[col] = out[col].ffill().bfill()

    return out

ts_full = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = create_features(ts_full, weekly)

n_tr = len(ts_train)
n_va = len(ts_val)
feat_train = ts_feat.iloc[:n_tr].dropna().copy()
feat_val   = ts_feat.iloc[n_tr:n_tr+n_va].dropna().copy()
feat_test  = ts_feat.iloc[n_tr+n_va:].dropna().copy()

feature_cols = [c for c in feat_train.columns if c not in ["ds", "y"]]
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Tabular split -- train: {len(feat_train)}, val: {len(feat_val)}, test: {len(feat_test)}")

## Baseline Approaches

Three simple baselines:
- **Naive** — repeat the last observed value
- **Seasonal Naive** — repeat the value from the same week one year ago
- **4-week Moving Average** — flat forecast at trailing monthly mean

In [ ]:
def eval_forecast(actual, predicted, name, is_holiday=None):
    actual, predicted = np.array(actual).flatten(), np.array(predicted).flatten()
    k = min(len(actual), len(predicted))
    actual, predicted = actual[:k], predicted[:k]
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.sum() else np.nan
    
    # Weighted MAE (competition metric -- holiday weeks get 5x weight)
    if is_holiday is not None:
        weights = np.where(is_holiday[:k], HOLIDAY_WEIGHT, 1.0)
        wmae = np.sum(weights * np.abs(actual - predicted)) / np.sum(weights)
    else:
        wmae = mae
    
    print(f"{name:35s} | MAE: {mae:14,.0f} | RMSE: {rmse:14,.0f} | WMAE: {wmae:14,.0f} | MAPE: {mape:6.2f}%")
    return {"model": name, "MAE": mae, "RMSE": rmse, "WMAE": wmae, "MAPE": mape}

results = []
test_holidays = ts_test["IsHoliday"].values if "IsHoliday" in ts_test.columns else None

# Naive
results.append(eval_forecast(ts_test["y"], np.full(TEST_WEEKS, ts_trainval["y"].iloc[-1]),
                             "Naive (last value)", test_holidays))

# Seasonal Naive (same week last year — 52 weeks ago)
if len(ts_trainval) >= 52:
    snaive_52 = ts_trainval["y"].iloc[-52 : -52 + TEST_WEEKS].values
    if len(snaive_52) == TEST_WEEKS:
        results.append(eval_forecast(ts_test["y"], snaive_52, "Seasonal Naive (52-week)", test_holidays))

# 4-week Moving Average
ma4 = np.full(TEST_WEEKS, ts_trainval["y"].iloc[-4:].mean())
results.append(eval_forecast(ts_test["y"], ma4, "4-week Moving Average", test_holidays))

print("\nBaselines computed.")

## LazyPredict Benchmark (Lag-Feature Tabular View)

LazyPredict benchmarks ~30 regressors on the tabular lag-feature representation. This scans model families quickly before we commit to tuning.

In [ ]:
X_tr_lp, y_tr_lp = feat_train[feature_cols], feat_train["y"]
X_va_lp, y_va_lp = feat_val[feature_cols],   feat_val["y"]

print(f"LazyPredict -- Train: {X_tr_lp.shape}, Val: {X_va_lp.shape}")

try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lazy_results, _ = lazy.fit(X_tr_lp, X_va_lp, y_tr_lp, y_va_lp)
    print("\nTop 15 models:")
    print(lazy_results.head(15).to_string())
except Exception as e:
    print(f"LazyPredict failed: {e}")
    lazy_results = None

## FLAML AutoML

In [ ]:
X_flaml = pd.concat([feat_train, feat_val])[feature_cols]
y_flaml = pd.concat([feat_train, feat_val])["y"]
X_test_flaml = feat_test[feature_cols]

print(f"FLAML -- Train: {X_flaml.shape}, Test: {X_test_flaml.shape}")

automl = AutoML()
automl.fit(
    X_flaml, y_flaml,
    task="regression",
    time_budget=FLAML_BUDGET,
    metric="rmse",
    verbose=0,
    seed=RANDOM_STATE,
)

print(f"\nBest estimator : {automl.best_estimator}")
print(f"Best config    : {automl.best_config}")

flaml_pred = automl.predict(X_test_flaml)
results.append(eval_forecast(feat_test["y"], flaml_pred,
                             f"FLAML ({automl.best_estimator})", test_holidays))

## MLForecast — Dedicated Time-Series Models

**Why MLForecast?** It wraps gradient-boosted tree models (LightGBM, XGBoost, sklearn) with built-in lag/rolling feature engineering, proper temporal cross-validation, and efficient multi-series support. For tabular-first forecasting at scale, it's one of the best tools available.

We train LightGBM and XGBoost within MLForecast's framework and compare against the baselines.

In [ ]:
# Prepare MLForecast format: (unique_id, ds, y)
mlf_data = ts_trainval[["ds", "y"]].copy()
mlf_data["unique_id"] = "walmart_total"
mlf_data = mlf_data[["unique_id", "ds", "y"]]

# Define MLForecast with lag features baked in
mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=200, learning_rate=0.05, num_leaves=31,
            random_state=RANDOM_STATE, verbosity=-1,
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=6,
            random_state=RANDOM_STATE, verbosity=0,
        ),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="W",
    lags=[1, 2, 4, 8, 13, 52],
    lag_transforms={
        1:  [("rolling_mean", 4), ("rolling_std", 4)],
        4:  [("rolling_mean", 13)],
        52: [("rolling_mean", 4)],
    },
    date_features=["week", "month", "quarter"],
)

mlf.fit(mlf_data)
mlf_fcst = mlf.predict(h=TEST_WEEKS)

print("MLForecast predictions:")
print(mlf_fcst.head())

for model_name in ["LightGBM", "XGBoost", "Ridge"]:
    if model_name in mlf_fcst.columns:
        pred = mlf_fcst[model_name].values
        results.append(eval_forecast(ts_test["y"].values, pred,
                                     f"MLF -- {model_name}", test_holidays))

In [ ]:
# Plot MLForecast forecasts vs actual
fig = go.Figure()
ctx = ts_trainval.iloc[-30:]
fig.add_trace(go.Scatter(x=ctx["ds"], y=ctx["y"], name="Train (context)", line=dict(color="blue", dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Actual", line=dict(color="black", width=2)))

colors = {"LightGBM": "green", "XGBoost": "orange", "Ridge": "purple"}
for model_name, color in colors.items():
    if model_name in mlf_fcst.columns:
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=mlf_fcst[model_name].values,
                                 name=f"MLF-{model_name}", line=dict(color=color)))

fig.update_layout(title="MLForecast Models vs Actual (8-week horizon)", template="plotly_white")
fig.show()

## Top 3 Model Selection

We rank all models by **WMAE** (Weighted MAE), the competition's actual metric, where holiday weeks are weighted 5× more than non-holiday weeks.

In [ ]:
results_df = pd.DataFrame(results).sort_values("WMAE").reset_index(drop=True)

print("=" * 90)
print("ALL MODELS -- Ranked by WMAE (Weighted MAE)")
print("=" * 90)
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'='*90}")
print("TOP 3 MODELS")
print(f"{'='*90}")
print(top3.to_string(index=False))

In [ ]:
fig = px.bar(results_df, x="model", y="WMAE",
             title="Model Comparison -- WMAE (lower is better, holiday-weighted)",
             color="WMAE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig.show()

fig2 = px.bar(results_df, x="model", y="MAE",
              title="Model Comparison -- MAE (lower is better)",
              color="MAE", color_continuous_scale="RdYlGn_r")
fig2.update_layout(xaxis_tickangle=-45, template="plotly_white")
fig2.show()

## Final Training & Evaluation of Top 3

In [ ]:
print("Top 3 models for final evaluation:")
for i, row in top3.iterrows():
    print(f"  {i+1}. {row['model']} -- WMAE: {row['WMAE']:,.0f}, MAE: {row['MAE']:,.0f}")

fig = go.Figure()
ctx = ts_trainval.iloc[-30:]
fig.add_trace(go.Scatter(x=ctx["ds"], y=ctx["y"], name="Train (context)", line=dict(color="blue", dash="dot")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Actual", line=dict(color="black", width=3)))

if len(feat_test) == len(ts_test):
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=flaml_pred[:len(ts_test)],
                             name=f"FLAML ({automl.best_estimator})", line=dict(color="green", dash="dash")))

fig.update_layout(title=f"Best model: {top3.iloc[0]['model']}",
                  xaxis_title="Date", yaxis_title="Total Weekly Sales", template="plotly_white")
fig.show()

## Error Analysis

We examine errors for the best tabular model, focusing on holiday vs non-holiday performance and systematic bias.

In [ ]:
if len(feat_test) > 0:
    actual = feat_test["y"].values
    pred   = flaml_pred[:len(actual)]
    errors = actual - pred

    print("Error Distribution (FLAML):")
    print(f"  Mean Error (bias) : {errors.mean():,.0f}")
    print(f"  Std Error         : {errors.std():,.0f}")
    print(f"  Max overpredict   : {errors.min():,.0f}")
    print(f"  Max underpredict  : {errors.max():,.0f}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(errors, bins=10, edgecolor="black", alpha=0.7)
    axes[0].axvline(0, color="red", ls="--")
    axes[0].set_title("Error Distribution")

    axes[1].plot(feat_test["ds"].values, errors, "o-", ms=6)
    axes[1].axhline(0, color="red", ls="--")
    axes[1].set_title("Errors Over Time")
    axes[1].tick_params(axis="x", rotation=45)

    axes[2].scatter(actual, pred, alpha=0.7, s=60)
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    axes[2].plot(lims, lims, "r--")
    axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
    axes[2].set_title("Actual vs Predicted")

    plt.tight_layout()
    plt.show()

    # Holiday vs non-holiday errors
    if "IsHoliday" in ts_test.columns:
        test_with_err = ts_test.copy()
        test_with_err["error"] = errors[:len(test_with_err)]
        test_with_err["abs_error"] = np.abs(test_with_err["error"])
        
        print("\nMAE by holiday status:")
        print(test_with_err.groupby("IsHoliday")["abs_error"].mean().to_string())
else:
    print("Insufficient test data for error analysis.")

## Interpretation & Insights

### Key Findings

1. **Strong annual seasonality** — Thanksgiving and Christmas produce the largest spikes. The pre-Christmas ramp starts in mid-November.
2. **Store type matters** — Type A stores generate ~60% of total sales despite being only ~50% of locations. Store size is a strong predictor.
3. **Markdowns shift demand** — MarkDown1 and MarkDown4 show the strongest correlation with sales uplift, especially around holidays.
4. **Economic indicators add signal** — CPI and Unemployment trend slowly, but fuel price spikes coincide with spending pullbacks.
5. **Holiday weeks are hardest to forecast** — the 5× weighted MAE in the competition metric reflects real business cost of holiday forecast errors.

### Which metric matters most?

The competition uses **WMAE** (holiday weeks 5× weighted). In practice, Walmart cares most about holiday forecast accuracy because holiday weeks account for disproportionate revenue and require supply chain pre-positioning weeks in advance.

## Limitations

1. **Short history** — only 143 weeks (~2.7 years) of training data limits our ability to learn inter-annual patterns.
2. **Aggregation** — we forecasted total Walmart sales; the competition requires per-store × per-department.
3. **Markdown leakage** — MarkDown values are unknown at forecast time; a production system needs a markdown forecast.
4. **No cross-series learning** — MLForecast supports multi-series, but we used single-series aggregate.
5. **Weekly resolution** — within-week patterns (day-of-week effects) are lost.
6. **US-centric holidays** — the model is specific to US retail calendar.

## How to Improve This Project

1. **Per-store × per-department models** — use MLForecast's multi-series support to train across 45×81 series simultaneously.
2. **Hierarchical reconciliation** — ensure store-level forecasts sum to regional and national totals.
3. **External regressors in MLForecast** — pass CPI, fuel price, and temperature as dynamic exogenous features.
4. **Cross-validation** — `mlf.cross_validation(h=8, n_windows=5)` for rolling-origin evaluation.
5. **LightGBM tuning** — Optuna hyperparameter search for num_leaves, max_depth, min_child_samples.
6. **Feature engineering** — year-over-year growth rate, days-until-holiday, rolling promotion intensity.
7. **Ensemble** — weighted average of top 3 models optimized on validation WMAE.
8. **Neural models** — try N-BEATS or TFT via NeuralForecast.

## Production Considerations

1. **Weekly retraining** — retrain every Monday with the latest week's data.
2. **Markup pipeline** — ingest markdown plans from merchandising team as exogenous features.
3. **Monitoring** — track WMAE per store; alert if holiday-week errors exceed historical 90th percentile.
4. **Forecast reconciliation** — store forecasts should sum to district/regional targets.
5. **Model registry** — version control models with metadata (training window, features used, WMAE).
6. **Serving** — expose forecasts via internal API consumed by supply chain and labor systems.
7. **Business translation** — convert unit sales forecasts to dollar forecasts using average price.

## Common Mistakes to Avoid

1. **Ignoring the weighted metric** — optimizing MAE instead of WMAE underweights holiday accuracy.
2. **Using markdown values at test time** — they are unknown in production.
3. **Random splitting** — shuffling weekly data destroys temporal autocorrelation.
4. **Wrong season_length** — using 7 (daily) instead of 52 (annual cycle for weekly data).
5. **Not handling negative sales** — department-level returns cause negative Weekly_Sales that distort averages.
6. **Ignoring store type** — pooling all stores without accounting for Type/Size differences.
7. **Overfitting to Thanksgiving spike** — only 2-3 Thanksgiving weeks in training makes the spike hard to learn.
8. **Using lag_52 without enough history** — you need at least 53 weeks to compute a 52-week lag.

## Mini Challenge / Exercises

1. **Per-store model** — pick Store 1 and build a single-store model. Compare to the aggregate.
2. **WMAE optimization** — create a custom loss function that weights holiday weeks 5× and train LightGBM with it.
3. **Store clustering** — use K-Means on (Size, CPI, Unemployment) to group stores, then train one model per cluster.
4. **Markdown analysis** — build a model that predicts sales lift from MarkDown1-5 for holiday vs non-holiday weeks.
5. **Ensemble stacking** — use top 3 models as base learners, fit a Ridge meta-learner on validation predictions.
6. **Temporal cross-validation** — evaluate with 4 rolling 8-week windows and report mean ± std WMAE.

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Walmart Store Sales** dataset (3 CSV files) from Kaggle
- Merged sales with store metadata (Type, Size) and economic features (Temperature, Fuel_Price, CPI, Unemployment)
- Aggregated 421K department records into a total-store weekly series
- Explored holiday effects (Super Bowl, Labor Day, Thanksgiving, Christmas), store types, and seasonality
- Built naive, seasonal naive, and moving-average baselines
- Benchmarked 30+ regressors via LazyPredict on tabular lag features with economic indicators
- Ran FLAML AutoML with a 2-minute time budget
- Trained MLForecast (LightGBM, XGBoost, Ridge) with built-in lag/rolling features
- Selected top 3 models from WMAE (holiday-weighted MAE)
- Analyzed errors by holiday status

### Key Takeaways
1. **Annual seasonality dominates** — Thanksgiving/Christmas weeks are extreme outliers that drive the competition metric.
2. **Store size is the single best predictor** — 0.9+ correlation with total sales.
3. **Tree-based models excel** — LightGBM and XGBoost consistently outperform linear models on this structured data.
4. **MLForecast simplifies production** — built-in lag features, proper temporal splits, and multi-series support.
5. **Weighted metrics matter** — optimizing for holiday accuracy changes model selection.

---
*Project 2 of 50 in the Time Series Forecasting Portfolio.*
*For educational purposes only -- always validate before production use.*